# Product Hunt Launch Success Analytics: Identifying Pre-Launch Factors That Influence Product Success

**University Business Analytics Project**  
**Dataset**: `synthetic_producthunt_10000.csv` (10,000 rows)

---

## 1. Project Introduction

### 1.1 Background of Product Hunt
Product Hunt is the primary launchpad for tech products, web apps, SaaS tools, and AI software. Every day at 12:00 AM PST (midnight), Product Hunt resets its daily leaderboard. Makers submit their products, and over the next 24 hours, the community votes, comments, and reviews them. Products that rank near the top of the daily leaderboard get prime homepage placement and significant industry exposure.

### 1.2 Why Launch Success Matters for Startups
For an early-stage startup or solo developer, a successful launch day can make a massive difference:
* **Immediate Traffic & Users**: A top-5 finish often brings thousands of visitors, sign-ups, and early paid customers within 48 hours.
* **Investor Attention**: Angel investors and venture capitalists regularly check Product Hunt's daily top products to spot high-traction teams early.
* **Social Proof & Press**: Winning a "Product of the Day" badge builds instant credibility and often leads to coverage in tech newsletters and blogs.

On the flip side, a weak launch means wasted preparation time and missed initial momentum.

### 1.3 Business Problem
Product launches on Product Hunt have huge variance in outcomes. Some tools get thousands of upvotes, while others barely get ten. Founders usually struggle with practical pre-launch decisions:
1. *Timing*: What hour of the day or day of the week works best?
2. *Content & Copy*: How detailed should the description be, and does adding a video actually help?
3. *Network*: How much does team size or having a high-follower "Hunter" boost votes?
4. *Category*: Does product category impact competition on the leaderboard?

Founders often rely on guessing or random blog posts instead of data. This project uses data analytics to evaluate which pre-launch choices actually correlate with launch performance.

### 1.4 Project Objectives
The main goal of this project is to build a clean analytics pipeline that evaluates pre-launch factors:
* Load, clean, and profile 10,000 Product Hunt launch records.
* Engineer practical features (launch time windows, weekend flags, combined social reach, log-transformed counts).
* Define a clear, balanced binary target variable (`SUCCESS`) based on median upvotes for future predictive modeling.

### 1.5 Scope & Business Value
This project focuses strictly on **pre-launch variables**—things founders can control or measure *before* pushing the launch button. Post-launch metrics (like comment speed or post-launch conversion rates) are excluded so the findings can be used as a practical pre-launch checklist for founders and growth marketers.


---

## 2. Dataset Construction & Synthetic Data Methodology

### 2.1 How the Data Was Collected
To analyze Product Hunt launches accurately, we built a dataset using a multi-step process:

1. **Product Hunt GraphQL API Pull**: We extracted raw historical post data directly from Product Hunt's GraphQL API. This gave us raw post metadata, vote counts, comment counts, maker and hunter follower numbers, media counts, and category tags.
2. **Data Cleaning & Standardization**: We converted launch timestamps into Pacific Standard Time (PST), calculated tagline and description character lengths, and formatted Maker/Hunter social network attributes.
3. **Synthetic Data Expansion with Gretel AI**: Because scraping the PH API directly hits rate limits and lacks user handle privacy, we expanded the scraped dataset to 10,000 observations using **Gretel AI** (approved by faculty for this coursework).

### 2.2 Why We Used Synthetic Data
Using synthetic data from Gretel AI solved two main problems while keeping our analysis valid:
* **Privacy Compliance**: It anonymizes individual user profiles, maker handles, and hunter names.
* **Adequate Sample Size ($N = 10,000$)**: 10,000 rows gives us plenty of statistical power for feature engineering, subgroup comparisons, and machine learning.
* **Preserved Statistical Distribution**: Gretel AI's generative tabular model was trained on the real Product Hunt API sample. It preserves the original data's distributions, column correlations (like hunter follower count vs upvotes), and variable ranges.

The resulting dataset, `synthetic_producthunt_10000.csv`, contains 10,000 rows and 17 initial features.


---

## 3. Import Libraries

Here we import standard Python libraries for data cleaning, basic math, and plotting.


In [1]:
# Data handling and numerical operations
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Stats utilities and warnings
import scipy.stats as stats
import warnings

# Clean plot aesthetic
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Ignore non-critical warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")


Libraries imported successfully.


---

## 4. Load Dataset

We load `synthetic_producthunt_10000.csv` and inspect its shape, column headers, and initial rows.


In [2]:
# Load CSV file into a pandas DataFrame
df = pd.read_csv('synthetic_producthunt_10000.csv')

# Check dataset dimensions
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Inspect raw column names
print("\nRaw Column Headers:")
print(df.columns.tolist())

# Preview first 5 rows
print("\nFirst 5 rows:")
df.head()


Dataset Shape: 10,000 rows, 17 columns

Raw Column Headers:
['    rank', 'votesCount', 'commentsCount', 'launch_hour', 'weekday', 'month', 'is_featured', 'description_length', 'tagline_length', 'topic_count', 'maker_count', 'maker_total_followers', 'hunter_followers_count', 'media_count', 'has_video', 'self_launched', 'primary_topic']

First 5 rows:


,rank,votesCount,commentsCount,launch_hour,weekday,month,is_featured,description_length,tagline_length,topic_count,maker_count,maker_total_followers,hunter_followers_count,media_count,has_video,self_launched,primary_topic
0,3,10.0,96.0,6,1,5,True,73,45,3,1,0,21548.0,7,True,False,Open Source
1,17,32.0,39.0,16,6,6,False,388,52,3,0,0,0.0,2,False,False,iOS
2,9,88.0,6.0,7,3,6,True,500,56,3,4,28,2323.0,3,False,False,Productivity
3,1,0.0,7.0,21,0,6,False,373,49,3,1,497,0.0,6,False,False,Design Tools
4,1,117.0,6.0,8,0,5,True,85,38,3,1,516,2641.0,9,False,False,Productivity


### What This Output Tells Us
1. **Dimensions**: We have 10,000 rows and 17 columns, giving us a large enough sample for statistical work.
2. **Column Names**: Notice that the first column `'    rank'` has extra leading spaces. We will clean this up in Section 6.
3. **Initial Data View**: The dataset includes upvote and comment counts (`votesCount`, `commentsCount`), timing info (`launch_hour`, `weekday`, `month`), copy lengths (`description_length`, `tagline_length`), social reach (`maker_total_followers`, `hunter_followers_count`), media metadata (`media_count`, `has_video`), and categories (`primary_topic`).


---

## 5. Data Understanding

Now we profile the dataset across five areas:
1. Column structure and memory usage (`df.info()`)
2. Summary statistics (`df.describe()`)
3. Missing values
4. Duplicate rows
5. Data types and unique value counts


### 5.1 Data Structure & Types (`df.info()`)


In [3]:
# Check column datatypes and non-null counts
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0       rank                10000 non-null  int64  
 1   votesCount              10000 non-null  float64
 2   commentsCount           10000 non-null  float64
 3   launch_hour             10000 non-null  int64  
 4   weekday                 10000 non-null  int64  
 5   month                   10000 non-null  int64  
 6   is_featured             10000 non-null  bool   
 7   description_length      10000 non-null  int64  
 8   tagline_length          10000 non-null  int64  
 9   topic_count             10000 non-null  int64  
 10  maker_count             10000 non-null  int64  
 11  maker_total_followers   10000 non-null  int64  
 12  hunter_followers_count  10000 non-null  float64
 13  media_count             10000 non-null  int64  
 14  has_video               10000 non-null 

#### Interpretation of `df.info()`
* **Memory Usage**: The dataset takes about 1.1 MB in memory, which is light and fast to process.
* **Completeness**: All 17 columns show 10,000 non-null values out of 10,000 entries. There are no missing values in this raw dataset.
* **Types**: We have 10 integers (`int64`), 3 floats (`float64`), 3 booleans (`bool`), and 1 text column (`object`).


### 5.2 Summary Statistics (`df.describe()`)


In [4]:
# Get summary statistics for numerical columns
df.describe().T


,count,mean,std,min,25%,50%,75%,max
rank,10000.0,11.4543,24.366351,1.0,1.0,6.0,12.00,222.0
votesCount,10000.0,123.5584,242.095245,0.0,14.0,70.0,139.00,2661.0
commentsCount,10000.0,24.2650,37.296811,0.0,0.0,11.0,29.00,342.0
launch_hour,10000.0,9.9949,5.538117,0.0,7.0,7.0,13.00,23.0
weekday,10000.0,2.9857,1.865541,0.0,1.0,3.0,5.00,6.0
month,10000.0,5.2620,1.244633,1.0,5.0,5.0,6.00,12.0
description_length,10000.0,356.3741,126.377308,48.0,260.0,390.0,467.00,500.0
tagline_length,10000.0,51.0197,9.439090,17.0,47.0,54.0,58.00,60.0
topic_count,10000.0,2.9792,1.151911,1.0,3.0,3.0,3.00,9.0
maker_count,10000.0,1.9244,1.933406,0.0,1.0,1.0,2.00,24.0


#### Interpretation of `df.describe()`
* **Upvotes (`votesCount`)**:
  * The mean upvote count is **123.56**, but the median is only **70.00**.
  * Votes range from **0** up to **2,661**. Because the mean is much higher than the median, votes are strongly **right-skewed**. A small percentage of top launches get the bulk of the votes.
* **Follower Reach**:
  * `hunter_followers_count`: Average is **14,335**, but the median is **1,894**, and the max is **170,171**. At least 25% of products have 0 hunter followers.
  * `maker_total_followers`: Median is **149.5**, with a max of **49,566**.
* **Text Lengths**:
  * `description_length`: Ranges from **48** to **500** characters (median is **390**). Most products use detailed descriptions.
  * `tagline_length`: Spans **17** to **60** characters (median is **54**).
* **Launch Hours**:
  * `launch_hour`: Median is **7 AM PST**, showing that launches cluster in the morning after the midnight PST reset.


### 5.3 Missing Values and Duplicate Rows


In [5]:
# Check missing value counts
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\nTotal duplicate rows: {duplicates}")


Missing values per column:
No missing values found.

Total duplicate rows: 0


#### Interpretation of Missing & Duplicate Analysis
* **Missing Data**: 0 missing values across all columns. No imputation is needed.
* **Duplicates**: 0 duplicate rows found. Every row is a unique launch entry.


### 5.4 Data Types & Unique Value Counts


In [6]:
# Inspect data types and unique value counts
pd.DataFrame({
    'Data Type': df.dtypes,
    'Unique Values': df.nunique(),
    'Sample Value': df.iloc[0]
})


,Data Type,Unique Values,Sample Value
rank,int64,178,3
votesCount,float64,721,10.0
commentsCount,float64,213,96.0
launch_hour,int64,24,6
weekday,int64,7,1
month,int64,12,5
is_featured,bool,2,True
description_length,int64,450,73
tagline_length,int64,44,45
topic_count,int64,8,3


#### Interpretation of Unique Value Profiles
* `primary_topic`: 71 unique categories (like Productivity, Mac, iOS, Developer Tools).
* `launch_hour`: 24 unique values (0 to 23 hours).
* `weekday`: 7 unique values (0=Monday to 6=Sunday).
* `is_featured`, `has_video`, `self_launched`: Boolean flags.


---

## 6. Data Cleaning & Preprocessing

In this step, we fix column name whitespace, clean up string values, adjust datatypes, and handle extreme values (outliers).


In [7]:
# Step 1: Strip extra spaces from column names
df.columns = df.columns.str.strip()
print("Cleaned column names:", df.columns.tolist())

# Step 2: Strip spaces from string values in primary_topic
df['primary_topic'] = df['primary_topic'].astype(str).str.strip()

# Step 3: Cast float count columns to integers
count_cols = ['votesCount', 'commentsCount', 'hunter_followers_count']
for col in count_cols:
    df[col] = df[col].astype(int)

# Step 4: Ensure boolean columns are proper bool type
bool_cols = ['is_featured', 'has_video', 'self_launched']
for col in bool_cols:
    df[col] = df[col].astype(bool)

print("\nUpdated column data types:")
print(df.dtypes)


Cleaned column names: ['rank', 'votesCount', 'commentsCount', 'launch_hour', 'weekday', 'month', 'is_featured', 'description_length', 'tagline_length', 'topic_count', 'maker_count', 'maker_total_followers', 'hunter_followers_count', 'media_count', 'has_video', 'self_launched', 'primary_topic']

Updated column data types:
rank                       int64
votesCount                 int64
commentsCount              int64
launch_hour                int64
weekday                    int64
month                      int64
is_featured                 bool
description_length         int64
tagline_length             int64
topic_count                int64
maker_count                int64
maker_total_followers      int64
hunter_followers_count     int64
media_count                int64
has_video                   bool
self_launched               bool
primary_topic             object
dtype: object


### Why Each Cleaning Step Was Done

1. **Stripping Column Names**: Fixing `'    rank'` to `'rank'` prevents typos and key errors when accessing columns in code.
2. **String Trimming**: Cleaning `primary_topic` ensures strings like `'Productivity '` and `'Productivity'` merge into a single category.
3. **Integer Casting**: Vote, comment, and follower counts are whole numbers. Casting them from floats to integers keeps data types clean and saves memory.

#### Outlier Strategy: Why We Do NOT Delete High Upvotes or Followers
In standard datasets, values far above the mean are sometimes dropped as outliers. But on Product Hunt, **extreme upvotes and high follower counts are real viral hits, not errors**.

* **Power-Law Dynamics**: Product Hunt engagement follows a power-law distribution where the top 5% of launches get the vast majority of votes and traffic.
* **Our Approach**: Deleting these rows would destroy genuine launch success stories. Instead of dropping them, we use **log-transformations** ($\log(1 + x)$) and **median-based splits** to handle skewness safely.


---

## 7. Feature Engineering

Now we derive practical business features from the raw data and define our target variable for future modeling.


In [8]:
# 1. Weekend Launch Flag (is_weekend)
# weekday coding: 0=Monday ... 5=Saturday, 6=Sunday
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x in [5, 6] else 0)

# 2. Launch Time Window (launch_window)
# Group hours into 3 practical launch windows relative to 00:00 PST reset
def get_launch_window(hour):
    if 0 <= hour <= 5:
        return 'PST Reset (00:00-05:59)'
    elif 6 <= hour <= 17:
        return 'Business Hours (06:00-17:59)'
    else:
        return 'Evening Off-Peak (18:00-23:59)'

df['launch_window'] = df['launch_hour'].apply(get_launch_window)

# 3. Total Pre-Launch Social Reach (total_social_reach)
# Combined follower reach of makers and hunter
df['total_social_reach'] = df['maker_total_followers'] + df['hunter_followers_count']

# 4. Log-Transformed Social Reach (log_social_reach)
df['log_social_reach'] = np.log1p(df['total_social_reach'])

# 5. Has Makers Flag (has_makers)
df['has_makers'] = df['maker_count'].apply(lambda x: 1 if x > 0 else 0)

# 6. Log-Transformed Votes (log_votesCount)
df['log_votesCount'] = np.log1p(df['votesCount'])

# 7. Binary Success Target (SUCCESS)
# Defined as 1 if votesCount >= median (70 upvotes), else 0
votes_median = df['votesCount'].median()
df['SUCCESS'] = (df['votesCount'] >= votes_median).astype(int)

# Preview newly engineered features
engineered_cols = [
    'votesCount', 'log_votesCount', 'SUCCESS', 'is_weekend', 
    'launch_window', 'total_social_reach', 'log_social_reach', 'has_makers'
]
print("Engineered Features Preview:")
df[engineered_cols].head(10)


Engineered Features Preview:


,votesCount,log_votesCount,SUCCESS,is_weekend,launch_window,total_social_reach,log_social_reach,has_makers
0,10,2.397895,0,0,Business Hours (06:00-17:59),21548,9.978085,1
1,32,3.496508,0,1,Business Hours (06:00-17:59),0,0.000000,0
2,88,4.488636,1,0,Business Hours (06:00-17:59),2351,7.763021,1
3,0,0.000000,0,0,Evening Off-Peak (18:00-23:59),497,6.210600,1
4,117,4.770685,1,0,Business Hours (06:00-17:59),3157,8.057694,1
5,141,4.955827,1,0,Evening Off-Peak (18:00-23:59),310,5.739793,1
6,54,4.007333,0,0,Business Hours (06:00-17:59),905,6.809039,1
7,35,3.583519,0,1,Business Hours (06:00-17:59),1323,7.188413,1
8,0,0.000000,0,1,Business Hours (06:00-17:59),7268,8.891374,1
9,46,3.850148,0,0,Business Hours (06:00-17:59),3844,8.254529,1


### Business Rationale for Engineered Features

#### Existing Features & Their Business Value:
* `launch_hour`: Exact hour of launch. Crucial because PH resets leaderboards at 00:00 PST. Launching early gives products a full 24 hours to gain votes.
* `weekday`: Day of the week. Helps evaluate weekday traffic vs weekend competition.
* `month`: Seasonality factor (e.g. tech launches slowing down near holidays).
* `maker_count`: Number of team members attached. More makers bring more combined social network reach.
* `description_length` & `tagline_length`: Measures copy detail vs quick punchiness.
* `media_count` & `has_video`: Visual assets. Video demos help explain complex software faster.
* `topic_count` & `primary_topic`: Product positioning and category competition density.

#### Rationale for New Derived Features:
1. `is_weekend`: Simple binary flag ($1$ for Sat/Sun, $0$ for weekdays) to test if launching on weekends makes it easier to rank high due to lower competition.
2. `launch_window`: Groups the 24 hours into 3 main operational windows ('PST Reset', 'Business Hours', 'Evening Off-Peak').
3. `total_social_reach` & `log_social_reach`: Combines maker and hunter follower counts into a single distribution reach metric. Taking $\log(1 + x)$ smooths out extreme right skewness.
4. `has_makers`: Flag indicating if a product has at least one listed maker ($1$ vs $0$).
5. `SUCCESS` (Binary Classification Target):
   * **Why Median Split (70 Upvotes)?**: The median upvote count in our dataset is **70 votes**. Products with $\ge 70$ votes get `SUCCESS = 1`, and products with $< 70$ votes get `SUCCESS = 0`.
   * **Why This Threshold Works**: Using the median split creates a clean **50% / 50% target balance**. It avoids class imbalance problems while separating above-average performing launches from below-average launches.


---

## 8. Final Quality Checks

We wrap up with four quality checks to make sure the dataset is clean and ready for analysis:
1. Check for missing values.
2. Verify dataset shape.
3. Check target variable balance.
4. Inspect summary stats for all engineered features.


In [9]:
# Check 1: Verify missing values count across entire dataframe
print(f"1. Remaining Missing Values: {df.isnull().sum().sum()}")

# Check 2: Verify final dataset dimensions
print(f"2. Final Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Check 3: Verify target variable class distribution
print("\n3. SUCCESS Target Class Distribution:")
print(df['SUCCESS'].value_counts(normalize=True) * 100)

# Check 4: Summary statistics of cleaned & engineered columns
print("\n4. Final Descriptive Statistics:")
df.describe().T[['mean', 'std', 'min', '50%', 'max']]


1. Remaining Missing Values: 0
2. Final Dataset Shape: 10,000 rows, 24 columns

3. SUCCESS Target Class Distribution:
1    50.42
0    49.58
Name: SUCCESS, dtype: float64

4. Final Descriptive Statistics:


,mean,std,min,50%,max
rank,11.454300,24.366351,1.0,6.000000,222.000000
votesCount,123.558400,242.095245,0.0,70.000000,2661.000000
commentsCount,24.265000,37.296811,0.0,11.000000,342.000000
launch_hour,9.994900,5.538117,0.0,7.000000,23.000000
weekday,2.985700,1.865541,0.0,3.000000,6.000000
month,5.262000,1.244633,1.0,5.000000,12.000000
description_length,356.374100,126.377308,48.0,390.000000,500.000000
tagline_length,51.019700,9.439090,17.0,54.000000,60.000000
topic_count,2.979200,1.151911,1.0,3.000000,9.000000
maker_count,1.924400,1.933406,0.0,1.000000,24.000000


### Summary of Quality Checks & Next Steps

#### Verification Results:
1. **Missing Values**: 0 missing values remaining across all 24 columns.
2. **Dataset Expansion**: Expanded from 17 raw variables to 24 clean features across 10,000 observations.
3. **Target Balance**: `SUCCESS` has a clean **50.4% / 49.6%** split ($5,042$ successful vs $4,958$ unsuccessful), so no synthetic resampling (SMOTE) is needed.
4. **Data Integrity**: All features have clean data types and reasonable ranges.

---

### Conclusion
This completes the setup, cleaning, and feature engineering for the Product Hunt Launch Success Analytics project (Sections 1 through 8).

The dataset is clean, properly formatted, and ready for **Exploratory Data Analysis (EDA)** and **Machine Learning Modeling**.
